# v1 — RoBERTa-base, 3 classes
Runtime → T4 GPU → Run all. ~50 min. See this folder's README.md.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Checkpoints go to Google Drive so a disconnect can be resumed. If the mount fails
# ("credential propagation was unsuccessful" happens with several Google accounts in one browser),
# fall back to the VM's disk: training still works, but a disconnect restarts from zero.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/btp_v1_roberta_base_3class'
except Exception as e:
    print(f'Drive mount failed ({e}); checkpoints stay on the VM.')
    DRIVE = '/content/btp_v1_roberta_base_3class'
!mkdir -p {DRIVE}
!test -d /content/BTP || git clone -q https://github.com/Abhijeet-SP/BTP.git /content/BTP
%cd /content/BTP
!git fetch -q origin && git reset -q --hard origin/main && git log --oneline -1
RESULTS = 'versions/v1_roberta_base_3class/results'
!pip -q install -U transformers datasets accelerate scikit-learn

In [ ]:
# ~8 min: downloads 2.3GB once, then a balanced 50k-per-class sample
!python prepare_data.py

In [ ]:
!python train_baseline_tfidf.py --results-dir {RESULTS}

In [ ]:
# ~42 min on a T4
!python finetune_roberta.py --name roberta_base_3class --batch-size 32 --results-dir {RESULTS} --ckpt-dir {DRIVE}/ckpt

In [ ]:
import os
assert os.path.exists('models/roberta_base_3class/config.json'), 'Training failed: scroll up. Re-run the cell to resume from the checkpoint.'
print('Training finished OK')

In [ ]:
import json
m = json.load(open(f'{RESULTS}/metrics.json'))
for k in ['tfidf_3class', 'roberta_base_3class', 'roberta_natural_prior']:
    if k in m:
        v = m[k]
        print(f"{k:28s} acc {v['accuracy']:.4f}  macro-F1 {v['macro_f1']:.4f}  "
              f"off-by-one {v['off_by_one_accuracy']:.4f}  MAE {v['mae_classes']:.3f}  QWK {v['quadratic_weighted_kappa']:.4f}")

In [ ]:
!zip -qr btp_v1_roberta_base_3class.zip models/roberta_base_3class versions/v1_roberta_base_3class/results && ls -lh btp_v1_roberta_base_3class.zip
!cp btp_v1_roberta_base_3class.zip {DRIVE}/
from google.colab import files
files.download('btp_v1_roberta_base_3class.zip')